# Notebook Unificado — Detección de deforestación en la Amazonía peruana

**Proyecto completo en un solo notebook:**
- **Parte A** (celdas 1–10): Extracción de datos desde Google Earth Engine
- **Parte B** (celdas 11–24): Entrenamiento de Attention U-Net + evaluación

## Flujo de ejecución

1. Corre **Parte A** completa (celdas 1–8) → lanza exportación a Drive
2. **Espera 30 min a 2 horas** hasta que GEE termine la exportación (revisa https://code.earthengine.google.com/tasks)
3. Corre **Parte A.2** (celdas 9–10) → reensambla VRT + genera parches
4. Corre **Parte B** completa → entrena y evalúa

## Mejoras v2 incorporadas

| # | Cambio | Dónde |
|---|---|---|
| 1 | Test con MapBiomas (mismo origen, split geográfico 70/15/15) | Celda 10 |
| 2 | 50 épocas para convergencia | Celdas 18, 20 |
| 3 | Búsqueda de umbral óptimo en validación | Celda 16, 21 |
| 4 | Loss híbrido BCE+Dice para Attention U-Net | Celdas 15, 20 |
| 5 | Carga robusta de parches (omite corruptos) | Celda 13 |

---

# PARTE A — Extracción de datos desde Google Earth Engine

## Celda 1 — Setup global (GEE + PyTorch + Drive)

In [ ]:
!pip install -q earthengine-api geemap rasterio==1.3.10

import ee, geemap
import os, glob, time, random, json, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Montar Drive una sola vez para todo el notebook
from google.colab import drive
drive.mount('/content/drive')

# GEE
GEE_PROJECT = 'feisty-mason-495316-e6'
try:
    ee.Initialize(project=GEE_PROJECT)
    print('GEE inicializado correctamente')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('GEE autenticado e inicializado')

# Rutas globales (Drive + local)
CARPETA_DRIVE      = '/content/drive/MyDrive/deforestacion_ucayali'
DIR_PARCHES_DRIVE  = f'{CARPETA_DRIVE}/parches'
DIR_PARCHES_LOCAL  = '/content/parches'
DIR_RESULTADOS = '/content/resultados'
os.makedirs(DIR_RESULTADOS, exist_ok=True)

# Device PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

## Celda 2 — Definición de AOI y fechas

In [ ]:
AOI = ee.Geometry.Rectangle([-75.0, -8.8, -74.0, -7.8])

FECHA_T1_INI, FECHA_T1_FIN = '2021-06-01', '2021-09-30'
FECHA_T2_INI, FECHA_T2_FIN = '2022-06-01', '2022-09-30'

Map = geemap.Map(center=[-8.3, -74.5], zoom=8)
Map.addLayer(AOI, {'color': 'red'}, 'AOI Ucayali')
Map

## Celda 3 — Mosaicos Sentinel-2 con máscara de nubes

In [ ]:
def enmascarar_nubes_s2(img):
    scl = img.select('SCL')
    mascara = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                .And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(mascara).divide(10000).copyProperties(img, ['system:time_start'])

def construir_mosaico(fecha_ini, fecha_fin, aoi):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(aoi).filterDate(fecha_ini, fecha_fin)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
           .map(enmascarar_nubes_s2))
    print(f'  {fecha_ini} → {fecha_fin}: {col.size().getInfo()} imágenes')
    return col.median().clip(aoi)

print('Construyendo mosaico T1 (2021)...')
mosaico_t1 = construir_mosaico(FECHA_T1_INI, FECHA_T1_FIN, AOI)
print('Construyendo mosaico T2 (2022)...')
mosaico_t2 = construir_mosaico(FECHA_T2_INI, FECHA_T2_FIN, AOI)

vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}
Map = geemap.Map(center=[-8.3, -74.5], zoom=9)
Map.addLayer(mosaico_t1, vis_rgb, 'T1 (2021) RGB')
Map.addLayer(mosaico_t2, vis_rgb, 'T2 (2022) RGB')
Map

## Celda 4 — Índices espectrales NDVI y BSI

In [ ]:
def calcular_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    bsi = img.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))',
        {'SWIR1': img.select('B11'), 'RED': img.select('B4'),
         'NIR': img.select('B8'), 'BLUE': img.select('B2')}
    ).rename('BSI')
    return img.addBands([ndvi, bsi])

mosaico_t1 = calcular_indices(mosaico_t1)
mosaico_t2 = calcular_indices(mosaico_t2)

vis_ndvi = {'bands': ['NDVI'], 'min': -0.2, 'max': 0.9,
            'palette': ['#a83232', '#dfdf3a', '#2eb82e', '#006400']}
Map = geemap.Map(center=[-8.3, -74.5], zoom=9)
Map.addLayer(mosaico_t1.select('NDVI'), vis_ndvi, 'NDVI 2021')
Map.addLayer(mosaico_t2.select('NDVI'), vis_ndvi, 'NDVI 2022')
Map

## Celda 5 — Etiquetas MapBiomas Amazonía Col 5

In [ ]:
MB_ASSET = 'projects/mapbiomas-raisg/public/collection5/mapbiomas_raisg_panamazonia_collection5_integration_v1'
mb = ee.Image(MB_ASSET)

cls_2021 = mb.select('classification_2021').clip(AOI)
cls_2022 = mb.select('classification_2022').clip(AOI)

bosque_2021_mb = cls_2021.eq(3).Or(cls_2021.eq(6))
bosque_2022_mb = cls_2022.eq(3).Or(cls_2022.eq(6))

etiqueta_mb = bosque_2021_mb.And(bosque_2022_mb.Not()).rename('deforestacion').toByte()
etiqueta_mb = etiqueta_mb.reproject(crs='EPSG:4326', scale=10).unmask(0)
bosque_mb_10m = bosque_2021_mb.reproject(crs='EPSG:4326', scale=10).unmask(0)

Map = geemap.Map(center=[-8.3, -74.5], zoom=10)
Map.addLayer(mosaico_t2, vis_rgb, 'T2 (2022) RGB')
Map.addLayer(etiqueta_mb.selfMask(), {'palette': ['#ff0000']}, 'Deforestación MapBiomas')
Map

## Celda 6 — Hansen GFC (referencia secundaria)

In [ ]:
gfc = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')

perdida_hansen_2022 = gfc.select('lossyear').eq(22).clip(AOI)
bosque_base_hansen = gfc.select('treecover2000').gte(30)
perdida_previa = gfc.select('lossyear').gt(0).And(gfc.select('lossyear').lte(21))
bosque_t1_hansen = bosque_base_hansen.And(perdida_previa.Not())

etiqueta_hs = perdida_hansen_2022.And(bosque_t1_hansen).rename('deforestacion').unmask(0).toByte()
etiqueta_hs = etiqueta_hs.reproject(crs='EPSG:4326', scale=10)

print('Hansen GFC procesado (queda en el stack como referencia)')

## Celda 7 — Estadísticas comparativas

In [ ]:
def contar(img, banda, scale=10):
    return img.reduceRegion(reducer=ee.Reducer.sum(),
                            geometry=AOI, scale=scale, maxPixels=1e10).getInfo()[banda]

n_mb = contar(etiqueta_mb, 'deforestacion', scale=10)
n_hs = contar(etiqueta_hs, 'deforestacion', scale=10)
n_total = contar(ee.Image(1).clip(AOI).rename('all'), 'all', scale=10)

print('=== Estadísticas de balance de clases ===')
print(f'Píxeles totales del AOI (10 m): {n_total:,.0f}\n')
print(f'MapBiomas Col 5 — GROUND TRUTH:')
print(f'  Píxeles positivos: {n_mb:,.0f}  ({100*n_mb/n_total:.3f}%)  ~{n_mb*100/10000:,.0f} ha\n')
print(f'Hansen GFC — referencia secundaria:')
print(f'  Píxeles positivos: {n_hs:,.0f}  ({100*n_hs/n_total:.3f}%)  ~{n_hs*100/10000:,.0f} ha')

## Celda 8 — Exportación a Drive

⚠ **Tras correr esta celda, espera 30 min – 2 h** hasta que GEE termine. Monitorea: https://code.earthengine.google.com/tasks

In [ ]:
bandas = ['B2', 'B3', 'B4', 'B8', 'NDVI', 'BSI']

stack = (mosaico_t1.select(bandas).rename([f'{b}_t1' for b in bandas])
         .addBands(mosaico_t2.select(bandas).rename([f'{b}_t2' for b in bandas]))
         .addBands(etiqueta_mb.rename('label_mb'))
         .addBands(etiqueta_hs.rename('label_hs')))

print('Bandas del stack:')
for b in stack.bandNames().getInfo():
    print(f'  {b}')

tarea = ee.batch.Export.image.toDrive(
    image=stack.toFloat(),
    description='stack_ucayali_2021_2022',
    folder='deforestacion_ucayali',
    fileNamePrefix='stack_ucayali_2021_2022',
    region=AOI,
    scale=10,
    maxPixels=1e10,
    fileFormat='GeoTIFF'
)
tarea.start()
print('\n✓ Exportación iniciada. Espera y revisa: https://code.earthengine.google.com/tasks')

## Celda 9 — Reensamblar fragmentos en VRT (correr DESPUÉS de que GEE termine)

In [ ]:
import rasterio
from rasterio.windows import Window

# Verificar que llegó el archivo
fragmentos = sorted(glob.glob(f'{CARPETA_DRIVE}/stack_ucayali_2021_2022-*.tif'))
print(f'Fragmentos encontrados: {len(fragmentos)}')
if len(fragmentos) == 0:
    raise FileNotFoundError('No hay fragmentos en Drive. ¿Terminó la exportación de GEE?')
for f in fragmentos:
    print(f'  {os.path.basename(f)} — {os.path.getsize(f)/1024/1024:.1f} MB')

def crear_vrt_manual(fragmentos, ruta_vrt):
    datasets = [rasterio.open(f) for f in fragmentos]
    bounds_list = [d.bounds for d in datasets]
    minx = min(b.left   for b in bounds_list)
    miny = min(b.bottom for b in bounds_list)
    maxx = max(b.right  for b in bounds_list)
    maxy = max(b.top    for b in bounds_list)

    ref = datasets[0]
    res_x = ref.transform.a
    res_y = -ref.transform.e
    width  = int(round((maxx - minx) / res_x))
    height = int(round((maxy - miny) / res_y))
    n_bandas = ref.count
    dtype = ref.dtypes[0]
    print(f'\nDimensiones reensambladas: {height} x {width} px, {n_bandas} bandas')

    xml = f'<VRTDataset rasterXSize="{width}" rasterYSize="{height}">\n'
    xml += f'  <SRS>{ref.crs.to_wkt()}</SRS>\n'
    xml += f'  <GeoTransform>{minx}, {res_x}, 0.0, {maxy}, 0.0, {-res_y}</GeoTransform>\n'

    dtype_vrt = {'float32':'Float32','float64':'Float64','uint8':'Byte',
                 'uint16':'UInt16','int16':'Int16','int32':'Int32'}[dtype]

    for b in range(1, n_bandas + 1):
        xml += f'  <VRTRasterBand dataType="{dtype_vrt}" band="{b}">\n'
        for ds, ruta in zip(datasets, fragmentos):
            offx = int(round((ds.bounds.left - minx) / res_x))
            offy = int(round((maxy - ds.bounds.top) / res_y))
            xml += f'    <SimpleSource>\n'
            xml += f'      <SourceFilename relativeToVRT="0">{ruta}</SourceFilename>\n'
            xml += f'      <SourceBand>{b}</SourceBand>\n'
            xml += f'      <SrcRect xOff="0" yOff="0" xSize="{ds.width}" ySize="{ds.height}"/>\n'
            xml += f'      <DstRect xOff="{offx}" yOff="{offy}" xSize="{ds.width}" ySize="{ds.height}"/>\n'
            xml += f'    </SimpleSource>\n'
        xml += f'  </VRTRasterBand>\n'
    xml += '</VRTDataset>\n'

    with open(ruta_vrt, 'w') as f: f.write(xml)
    for d in datasets: d.close()
    return ruta_vrt

RUTA_VRT = f'{CARPETA_DRIVE}/stack_ucayali_2021_2022.vrt'
crear_vrt_manual(fragmentos, RUTA_VRT)

with rasterio.open(RUTA_VRT) as src:
    print(f'\n✓ VRT funciona: {src.height}×{src.width} px, {src.count} bandas')

## Celda 10 — Generación de parches (split geográfico 70/15/15 con MapBiomas)

In [ ]:
import rasterio
from rasterio.windows import Window
RUTA_VRT = f'{CARPETA_DRIVE}/stack_ucayali_2021_2022.vrt'

TAM_PARCHE = 256
STRIDE = 256
MIN_PCT_POS = 0.001
LABEL_IDX = 12  # banda label_mb (MapBiomas) en el stack

for sp in ['train', 'val', 'test']:
    os.makedirs(f'{DIR_PARCHES_DRIVE}/{sp}', exist_ok=True)

with rasterio.open(RUTA_VRT) as src:
    H, W = src.height, src.width
    print(f'Imagen: {H} × {W} px, {src.count} bandas')

    fila_train_fin = int(H * 0.70)
    fila_val_fin   = int(H * 0.85)

    cont = {'train': 0, 'val': 0, 'test': 0,
            'descartados_nan': 0, 'descartados_balance': 0}

    for fila in range(0, H - TAM_PARCHE + 1, STRIDE):
        for col in range(0, W - TAM_PARCHE + 1, STRIDE):
            ventana = Window(col, fila, TAM_PARCHE, TAM_PARCHE)
            parche = src.read(window=ventana)

            if np.isnan(parche[:12]).mean() > 0.20:
                cont['descartados_nan'] += 1
                continue

            if fila + TAM_PARCHE <= fila_train_fin:
                split = 'train'
            elif fila + TAM_PARCHE <= fila_val_fin:
                split = 'val'
            else:
                split = 'test'

            etiqueta = parche[LABEL_IDX]
            pct_pos = np.nanmean(etiqueta)
            if pct_pos < MIN_PCT_POS and np.random.random() > 0.10:
                cont['descartados_balance'] += 1
                continue

            parche_final = np.concatenate([parche[:12], etiqueta[None,:,:]], axis=0)
            parche_final = np.nan_to_num(parche_final, nan=0.0).astype(np.float32)

            ruta = f'{DIR_PARCHES_DRIVE}/{split}/parche_{fila:06d}_{col:06d}.npy'
            np.save(ruta, parche_final)
            cont[split] += 1

print('\n--- Resumen ---')
for k, v in cont.items():
    print(f'  {k}: {v}')

---
# PARTE B — Entrenamiento Attention U-Net + Evaluación

## Celda 11 — Copia robusta a disco local + validación de integridad

Copia parches desde Drive a `/content/parches` con reintentos automáticos (Drive a veces se desconecta) y elimina archivos corruptos.

In [ ]:
def copiar_con_reintentos(src, dst, intentos=3, espera=2):
    for i in range(intentos):
        try:
            shutil.copy(src, dst)
            return True
        except OSError:
            if i < intentos - 1:
                time.sleep(espera)
            else:
                return False

def copiar_parches():
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{DIR_PARCHES_LOCAL}/{split}', exist_ok=True)
        rutas = sorted(glob.glob(f'{DIR_PARCHES_DRIVE}/{split}/*.npy'))
        print(f'Copiando {split}: {len(rutas)} parches...')
        t0 = time.time()
        fallos = 0
        for r in rutas:
            destino = f'{DIR_PARCHES_LOCAL}/{split}/{os.path.basename(r)}'
            if not os.path.exists(destino):
                if not copiar_con_reintentos(r, destino):
                    fallos += 1
        print(f'  ✓ {time.time()-t0:.1f} s | fallos: {fallos}')

copiar_parches()

# Limpiar archivos corruptos
print('\n--- Validación de integridad ---')
total_corruptos = 0
for split in ['train', 'val', 'test']:
    corruptos = []
    rutas = sorted(glob.glob(f'{DIR_PARCHES_LOCAL}/{split}/*.npy'))
    for r in rutas:
        try:
            arr = np.load(r, allow_pickle=False)
            if arr.shape != (13, 256, 256):
                corruptos.append(r)
        except Exception:
            corruptos.append(r)
    for r in corruptos:
        os.remove(r)
    total_corruptos += len(corruptos)
    n_ok = len(rutas) - len(corruptos)
    print(f'  {split}: {n_ok} OK | {len(corruptos)} eliminados')

print(f'\nTotal corruptos eliminados: {total_corruptos}')

## Celda 12 — Inspección visual de parches (QA)

In [ ]:
parches_train = sorted(glob.glob(f'{DIR_PARCHES_LOCAL}/train/*.npy'))
print(f'Parches en train: {len(parches_train)}')

ejemplos = []
for p in random.sample(parches_train, min(80, len(parches_train))):
    arr = np.load(p)
    if arr[12].mean() > 0.03:
        ejemplos.append(p)
    if len(ejemplos) >= 3: break

if ejemplos:
    fig, axes = plt.subplots(len(ejemplos), 5, figsize=(20, 4*len(ejemplos)))
    if len(ejemplos) == 1: axes = axes[None, :]

    for i, ruta in enumerate(ejemplos):
        arr = np.load(ruta)
        rgb_t1 = np.clip(np.stack([arr[2], arr[1], arr[0]], axis=-1)/0.3, 0, 1)
        rgb_t2 = np.clip(np.stack([arr[8], arr[7], arr[6]], axis=-1)/0.3, 0, 1)
        ndvi_diff = arr[10] - arr[4]
        bsi_diff  = arr[11] - arr[5]
        label = arr[12]
        axes[i,0].imshow(rgb_t1); axes[i,0].set_title('RGB T1 (2021)')
        axes[i,1].imshow(rgb_t2); axes[i,1].set_title('RGB T2 (2022)')
        axes[i,2].imshow(ndvi_diff, cmap='RdYlGn', vmin=-0.5, vmax=0.5); axes[i,2].set_title('ΔNDVI')
        axes[i,3].imshow(bsi_diff,  cmap='YlOrBr', vmin=-0.3, vmax=0.3); axes[i,3].set_title('ΔBSI')
        axes[i,4].imshow(label, cmap='Reds'); axes[i,4].set_title(f'Etiqueta ({label.mean()*100:.1f}%)')
        for ax in axes[i]: ax.axis('off')
    plt.tight_layout(); plt.show()

## Celda 13 — Dataset y DataLoaders (carga robusta)

In [ ]:
class DatasetDeforestacion(Dataset):
    """Dataset robusto: omite parches que fallan al cargarse en runtime."""
    def __init__(self, rutas, media=None, std=None, augment=False):
        self.rutas = rutas
        self.media = media
        self.std = std
        self.augment = augment

    def __len__(self): return len(self.rutas)

    def __getitem__(self, idx):
        for offset in range(len(self.rutas)):
            try:
                ruta = self.rutas[(idx + offset) % len(self.rutas)]
                arr = np.load(ruta, allow_pickle=False)
                if arr.shape != (13, 256, 256):
                    continue
                break
            except Exception:
                continue
        else:
            raise RuntimeError('Ningún parche válido en el dataset')

        x = arr[:12].astype(np.float32)
        y = arr[12].astype(np.float32)

        if self.media is not None:
            x = (x - self.media[:, None, None]) / (self.std[:, None, None] + 1e-6)

        if self.augment:
            if random.random() < 0.5:
                x = x[:, :, ::-1].copy(); y = y[:, ::-1].copy()
            if random.random() < 0.5:
                x = x[:, ::-1, :].copy(); y = y[::-1, :].copy()
            k = random.randint(0, 3)
            x = np.rot90(x, k, axes=(1,2)).copy(); y = np.rot90(y, k).copy()

        return torch.from_numpy(x), torch.from_numpy(y).unsqueeze(0)


rutas_train = sorted(glob.glob(f'{DIR_PARCHES_LOCAL}/train/*.npy'))
rutas_val   = sorted(glob.glob(f'{DIR_PARCHES_LOCAL}/val/*.npy'))
rutas_test  = sorted(glob.glob(f'{DIR_PARCHES_LOCAL}/test/*.npy'))
print(f'Train: {len(rutas_train)} | Val: {len(rutas_val)} | Test: {len(rutas_test)}')

print('\nCalculando estadísticas de normalización sobre train...')
muestras = []
for r in random.sample(rutas_train, min(200, len(rutas_train))):
    try:
        muestras.append(np.load(r, allow_pickle=False)[:12])
    except Exception:
        pass
muestras = np.stack(muestras)
media = muestras.mean(axis=(0,2,3)).astype(np.float32)
std   = muestras.std(axis=(0,2,3)).astype(np.float32)
print('Media por canal:', np.round(media, 3))
print('Std por canal:  ', np.round(std, 3))
np.savez(f'{DIR_RESULTADOS}/normalizacion.npz', media=media, std=std)

BATCH = 8
ds_train = DatasetDeforestacion(rutas_train, media, std, augment=True)
ds_val   = DatasetDeforestacion(rutas_val,   media, std, augment=False)
ds_test  = DatasetDeforestacion(rutas_test,  media, std, augment=False)

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## Celda 14 — Arquitecturas: U-Net y Attention U-Net

In [ ]:
class DobleConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)


class UNet(nn.Module):
    """U-Net estándar — baseline."""
    def __init__(self, c_in=12, c_out=1, base=32):
        super().__init__()
        self.e1 = DobleConv(c_in, base)
        self.e2 = DobleConv(base, base*2)
        self.e3 = DobleConv(base*2, base*4)
        self.e4 = DobleConv(base*4, base*8)
        self.bottleneck = DobleConv(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.d4 = DobleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.d3 = DobleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.d2 = DobleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.d1 = DobleConv(base*2, base)
        self.out = nn.Conv2d(base, c_out, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.d4(torch.cat([self.up4(b), e4], 1))
        d3 = self.d3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


class AttentionGate(nn.Module):
    """Attention Gate de Oktay et al. (2018), referenciada por John & Zhang (2022)."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g); x1 = self.W_x(x)
        psi = self.psi(self.relu(g1 + x1))
        return x * psi


class AttentionUNet(nn.Module):
    def __init__(self, c_in=12, c_out=1, base=32):
        super().__init__()
        self.e1 = DobleConv(c_in, base)
        self.e2 = DobleConv(base, base*2)
        self.e3 = DobleConv(base*2, base*4)
        self.e4 = DobleConv(base*4, base*8)
        self.bottleneck = DobleConv(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.ag4 = AttentionGate(F_g=base*8, F_l=base*8, F_int=base*4)
        self.d4 = DobleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.ag3 = AttentionGate(F_g=base*4, F_l=base*4, F_int=base*2)
        self.d3 = DobleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.ag2 = AttentionGate(F_g=base*2, F_l=base*2, F_int=base)
        self.d2 = DobleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.ag1 = AttentionGate(F_g=base, F_l=base, F_int=base//2)
        self.d1 = DobleConv(base*2, base)
        self.out = nn.Conv2d(base, c_out, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        u4 = self.up4(b);  e4_at = self.ag4(g=u4, x=e4); d4 = self.d4(torch.cat([u4, e4_at], 1))
        u3 = self.up3(d4); e3_at = self.ag3(g=u3, x=e3); d3 = self.d3(torch.cat([u3, e3_at], 1))
        u2 = self.up2(d3); e2_at = self.ag2(g=u2, x=e2); d2 = self.d2(torch.cat([u2, e2_at], 1))
        u1 = self.up1(d2); e1_at = self.ag1(g=u1, x=e1); d1 = self.d1(torch.cat([u1, e1_at], 1))
        return self.out(d1)


for nombre, modelo in [('UNet', UNet()), ('AttentionUNet', AttentionUNet())]:
    x = torch.randn(2, 12, 256, 256)
    y = modelo(x)
    n_params = sum(p.numel() for p in modelo.parameters())
    print(f'{nombre}: salida {y.shape}, params {n_params:,}')

## Celda 15 — Funciones de pérdida (incluye loss híbrido BCE+Dice) y métricas

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps=1.0):
        super().__init__(); self.eps = eps
    def forward(self, logits, y):
        prob = torch.sigmoid(logits)
        inter = (prob * y).sum(dim=(2,3))
        union = prob.sum(dim=(2,3)) + y.sum(dim=(2,3))
        dice = (2*inter + self.eps) / (union + self.eps)
        return 1 - dice.mean()


class BCEDiceLoss(nn.Module):
    """Loss híbrido: combina BCE (pixel-wise con pos_weight) y Dice (region-wise)."""
    def __init__(self, pos_weight=None, dice_eps=1.0, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.dice = DiceLoss(eps=dice_eps)
        self.bce_w = bce_weight
        self.dice_w = dice_weight
    def forward(self, logits, y):
        return self.bce_w * self.bce(logits, y) + self.dice_w * self.dice(logits, y)


def calcular_metricas(logits, y, umbral=0.5):
    pred = (torch.sigmoid(logits) > umbral).float()
    tp = ((pred == 1) & (y == 1)).sum().item()
    fp = ((pred == 1) & (y == 0)).sum().item()
    fn = ((pred == 0) & (y == 1)).sum().item()
    tn = ((pred == 0) & (y == 0)).sum().item()
    return tp, fp, fn, tn

def metricas_finales(tp, fp, fn, tn, eps=1e-7):
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2*precision*recall / (precision + recall + eps)
    iou       = tp / (tp + fp + fn + eps)
    accuracy  = (tp + tn) / (tp + fp + fn + tn + eps)
    return {'precision': precision, 'recall': recall, 'f1_dice': f1,
            'iou': iou, 'accuracy': accuracy}

## Celda 16 — Loop de entrenamiento + búsqueda de umbral óptimo

In [ ]:
def entrenar(modelo, loss_fn, nombre_exp, epocas=50, lr=1e-3):
    modelo = modelo.to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)

    historial = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_f1': []}
    mejor_iou = -1.0
    ruta_modelo = f'{DIR_RESULTADOS}/{nombre_exp}_best.pt'

    for ep in range(epocas):
        modelo.train()
        loss_acum = 0.0; n = 0
        t0 = time.time()
        for x, y in dl_train:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = modelo(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
            loss_acum += loss.item() * x.size(0); n += x.size(0)
        train_loss = loss_acum / n

        modelo.eval()
        loss_acum = 0.0; n = 0; TP=FP=FN=TN=0
        with torch.no_grad():
            for x, y in dl_val:
                x, y = x.to(device), y.to(device)
                logits = modelo(x)
                loss = loss_fn(logits, y)
                loss_acum += loss.item() * x.size(0); n += x.size(0)
                tp, fp, fn_, tn = calcular_metricas(logits, y, umbral=0.5)
                TP+=tp; FP+=fp; FN+=fn_; TN+=tn
        val_loss = loss_acum / n
        m = metricas_finales(TP, FP, FN, TN)

        sched.step(m['iou'])
        historial['train_loss'].append(train_loss)
        historial['val_loss'].append(val_loss)
        historial['val_iou'].append(m['iou'])
        historial['val_f1'].append(m['f1_dice'])

        elapsed = time.time() - t0
        print(f"[{nombre_exp}] Ep {ep+1:02d}/{epocas} | tr_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
              f"| IoU {m['iou']:.4f} | F1 {m['f1_dice']:.4f} | P {m['precision']:.3f} R {m['recall']:.3f} | {elapsed:.0f}s")

        if m['iou'] > mejor_iou:
            mejor_iou = m['iou']
            torch.save(modelo.state_dict(), ruta_modelo)
            print(f'  → guardado (IoU mejorado a {mejor_iou:.4f})')

    with open(f'{DIR_RESULTADOS}/{nombre_exp}_historial.json', 'w') as f:
        json.dump(historial, f, indent=2)

    return ruta_modelo, historial


def buscar_umbral_optimo(modelo, ruta_pesos, umbrales=None):
    """Escanea umbrales en validación, devuelve el que maximiza F1."""
    if umbrales is None:
        umbrales = np.arange(0.10, 0.71, 0.05)

    modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
    modelo.to(device).eval()

    todos_logits, todos_y = [], []
    with torch.no_grad():
        for x, y in dl_val:
            x, y = x.to(device), y.to(device)
            logits = modelo(x)
            todos_logits.append(torch.sigmoid(logits).cpu())
            todos_y.append(y.cpu())
    probs = torch.cat(todos_logits, dim=0)
    ys    = torch.cat(todos_y, dim=0)

    tabla = []
    for u in umbrales:
        pred = (probs > u).float()
        tp = ((pred == 1) & (ys == 1)).sum().item()
        fp = ((pred == 1) & (ys == 0)).sum().item()
        fn = ((pred == 0) & (ys == 1)).sum().item()
        tn = ((pred == 0) & (ys == 0)).sum().item()
        m = metricas_finales(tp, fp, fn, tn)
        m['umbral'] = float(u)
        tabla.append(m)

    mejor = max(tabla, key=lambda d: d['f1_dice'])
    print(f'\n--- Búsqueda de umbral óptimo (validación) ---')
    print(f'{"Umbral":>8} {"Precision":>10} {"Recall":>10} {"F1":>10} {"IoU":>10}')
    for m in tabla:
        marker = ' ← mejor' if m['umbral'] == mejor['umbral'] else ''
        print(f"{m['umbral']:>8.2f} {m['precision']:>10.4f} {m['recall']:>10.4f} "
              f"{m['f1_dice']:>10.4f} {m['iou']:>10.4f}{marker}")
    return mejor['umbral'], mejor, tabla

## Celda 17 — Experimento 1: Baseline U-Net + BCE

In [ ]:
# Ajustar pos_weight al nuevo balance (~5% positivos)
pos_weight = torch.tensor([19.0], device=device)

In [ ]:
import shutil, os, glob, time

DIR_PARCHES_DRIVE = '/content/drive/MyDrive/deforestacion_ucayali/parches'
DIR_PARCHES_LOCAL = '/content/parches'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{DIR_PARCHES_LOCAL}/{split}', exist_ok=True)
    rutas = sorted(glob.glob(f'{DIR_PARCHES_DRIVE}/{split}/*.npy'))
    print(f'Copiando {split}: {len(rutas)} parches...')
    t0 = time.time()
    for r in rutas:
        dst = f'{DIR_PARCHES_LOCAL}/{split}/{os.path.basename(r)}'
        if not os.path.exists(dst):
            shutil.copy(r, dst)
    print(f'  ✓ {time.time()-t0:.0f}s')

print('Copia completa. Ahora actualiza DIR_PARCHES_LOCAL en el Dataset.')

In [ ]:
import os, glob
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f'/content/parches/{split}/*.npy'))
    print(f'{split}: {n} parches')

In [ ]:
pos_weight = torch.tensor([60.0], device=device)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

modelo_baseline = UNet(c_in=12, c_out=1, base=32)
ruta_baseline, hist_baseline = entrenar(
    modelo_baseline, bce, 'unet_baseline_v2', epocas=120, lr=1e-3
)

## Celda 18 — Experimento 2: Attention U-Net + Loss híbrido BCE+Dice

In [ ]:
loss_hibrido = BCEDiceLoss(
    pos_weight=torch.tensor([60.0], device=device),
    dice_eps=1.0,
    bce_weight=0.5,
    dice_weight=0.5
)

modelo_attn = AttentionUNet(c_in=12, c_out=1, base=32)
ruta_attn, hist_attn = entrenar(
    modelo_attn, loss_hibrido, 'attention_unet_bcedice_v2', epocas=120, lr=1e-3
)

## Celda 19 — Búsqueda de umbral óptimo para cada modelo

In [ ]:
print('=' * 70)
print('BÚSQUEDA DE UMBRAL ÓPTIMO — Baseline U-Net + BCE')
print('=' * 70)
umbral_baseline, m_val_baseline, tabla_baseline = buscar_umbral_optimo(
    UNet(c_in=12, c_out=1, base=32), ruta_baseline
)

print('\n' + '=' * 70)
print('BÚSQUEDA DE UMBRAL ÓPTIMO — Attention U-Net + BCE+Dice')
print('=' * 70)
umbral_attn, m_val_attn, tabla_attn = buscar_umbral_optimo(
    AttentionUNet(c_in=12, c_out=1, base=32), ruta_attn
)

with open(f'{DIR_RESULTADOS}/umbrales_optimos.json', 'w') as f:
    json.dump({
        'baseline':       {'umbral_optimo': umbral_baseline, 'tabla_val': tabla_baseline},
        'attention_unet': {'umbral_optimo': umbral_attn,     'tabla_val': tabla_attn}
    }, f, indent=2)

print(f'\n→ Umbral óptimo Baseline: {umbral_baseline:.2f}')
print(f'→ Umbral óptimo AttnUNet: {umbral_attn:.2f}')

## Celda 20 — Evaluación final en TEST (umbral default + óptimo)

In [ ]:
def evaluar_test(modelo, ruta_pesos, umbral=0.5):
    modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
    modelo.to(device).eval()
    TP=FP=FN=TN=0
    with torch.no_grad():
        for x, y in dl_test:
            x, y = x.to(device), y.to(device)
            logits = modelo(x)
            tp, fp, fn_, tn = calcular_metricas(logits, y, umbral=umbral)
            TP+=tp; FP+=fp; FN+=fn_; TN+=tn
    return metricas_finales(TP, FP, FN, TN)


print('--- Evaluación TEST con umbral default (0.5) ---')
m_baseline_05 = evaluar_test(UNet(c_in=12, c_out=1, base=32),         ruta_baseline, umbral=0.5)
m_attn_05     = evaluar_test(AttentionUNet(c_in=12, c_out=1, base=32), ruta_attn,     umbral=0.5)

print('--- Evaluación TEST con umbral óptimo ---')
m_baseline_opt = evaluar_test(UNet(c_in=12, c_out=1, base=32),         ruta_baseline, umbral=umbral_baseline)
m_attn_opt     = evaluar_test(AttentionUNet(c_in=12, c_out=1, base=32), ruta_attn,     umbral=umbral_attn)

print('\n' + '=' * 90)
print('RESULTADOS EN TEST')
print('=' * 90)
print(f"{'Métrica':<12} {'Baseline (u=0.50)':>18} {'AttnUNet (u=0.50)':>18} "
      f"{'Baseline (u=opt)':>18} {'AttnUNet (u=opt)':>18}")
print('-' * 90)
for k in ['iou', 'f1_dice', 'precision', 'recall', 'accuracy']:
    print(f"{k:<12} {m_baseline_05[k]:>18.4f} {m_attn_05[k]:>18.4f} "
          f"{m_baseline_opt[k]:>18.4f} {m_attn_opt[k]:>18.4f}")

print(f'\nUmbral óptimo Baseline: {umbral_baseline:.2f}')
print(f'Umbral óptimo AttnUNet: {umbral_attn:.2f}')

delta_f1  = m_attn_opt['f1_dice'] - m_baseline_opt['f1_dice']
delta_iou = m_attn_opt['iou']     - m_baseline_opt['iou']
mejora_rel_f1  = 100 * delta_f1  / max(m_baseline_opt['f1_dice'], 1e-6)
mejora_rel_iou = 100 * delta_iou / max(m_baseline_opt['iou'], 1e-6)
print(f'\n--- Mejora relativa (AttnUNet+BCE+Dice vs Baseline, umbrales óptimos) ---')
print(f'  ΔF1:  {delta_f1:+.4f}  ({mejora_rel_f1:+.1f}% relativo)')
print(f'  ΔIoU: {delta_iou:+.4f}  ({mejora_rel_iou:+.1f}% relativo)')

with open(f'{DIR_RESULTADOS}/resultados_test.json', 'w') as f:
    json.dump({
        'umbrales_optimos': {'baseline': umbral_baseline, 'attention_unet': umbral_attn},
        'test_umbral_0.5':     {'baseline': m_baseline_05,  'attention_unet': m_attn_05},
        'test_umbral_optimo':  {'baseline': m_baseline_opt, 'attention_unet': m_attn_opt}
    }, f, indent=2)

## Celda 21 — Visualización de predicciones lado a lado

In [ ]:
modelo_baseline = UNet(c_in=12, c_out=1, base=32)
modelo_baseline.load_state_dict(torch.load(ruta_baseline, map_location=device))
modelo_baseline.to(device).eval()

modelo_attn = AttentionUNet(c_in=12, c_out=1, base=32)
modelo_attn.load_state_dict(torch.load(ruta_attn, map_location=device))
modelo_attn.to(device).eval()

ejemplos = []
for r in rutas_test:
    try:
        arr = np.load(r, allow_pickle=False)
        if arr[12].mean() > 0.05:
            ejemplos.append(r)
        if len(ejemplos) >= 4:
            break
    except Exception:
        continue

fig, axes = plt.subplots(len(ejemplos), 6, figsize=(24, 4*len(ejemplos)))
if len(ejemplos) == 1: axes = axes[None, :]

with torch.no_grad():
    for i, ruta in enumerate(ejemplos):
        arr = np.load(ruta, allow_pickle=False)
        x_raw = arr[:12]; y = arr[12]
        x_norm = ((x_raw - media[:,None,None]) / (std[:,None,None] + 1e-6)).astype(np.float32)
        x_t = torch.from_numpy(x_norm).unsqueeze(0).to(device)

        prob_b = torch.sigmoid(modelo_baseline(x_t))[0,0].cpu().numpy()
        prob_a = torch.sigmoid(modelo_attn(x_t))[0,0].cpu().numpy()
        pred_b_opt = (prob_b > umbral_baseline).astype(np.float32)
        pred_a_opt = (prob_a > umbral_attn).astype(np.float32)

        rgb_t2 = np.clip(np.stack([x_raw[8], x_raw[7], x_raw[6]], axis=-1)/0.3, 0, 1)

        axes[i,0].imshow(rgb_t2);                              axes[i,0].set_title('RGB T2 (2022)')
        axes[i,1].imshow(y, cmap='Reds');                      axes[i,1].set_title('Ground Truth')
        axes[i,2].imshow(prob_b, cmap='Reds', vmin=0, vmax=1); axes[i,2].set_title('Baseline (prob)')
        axes[i,3].imshow(pred_b_opt, cmap='Reds');             axes[i,3].set_title(f'Baseline (u={umbral_baseline:.2f})')
        axes[i,4].imshow(prob_a, cmap='Reds', vmin=0, vmax=1); axes[i,4].set_title('AttnUNet (prob)')
        axes[i,5].imshow(pred_a_opt, cmap='Reds');             axes[i,5].set_title(f'AttnUNet (u={umbral_attn:.2f})')
        for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/comparacion_visual.png', dpi=120, bbox_inches='tight')
plt.show()

## Celda 22 — Curvas de aprendizaje + umbral vs F1

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

ax = axes[0,0]
ax.plot(hist_baseline['train_loss'], label='Baseline train', linewidth=2)
ax.plot(hist_baseline['val_loss'],   label='Baseline val',   linewidth=2)
ax.plot(hist_attn['train_loss'],     label='AttnUNet train', linewidth=2, linestyle='--')
ax.plot(hist_attn['val_loss'],       label='AttnUNet val',   linewidth=2, linestyle='--')
ax.set_title('Loss por época'); ax.set_xlabel('Época'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0,1]
ax.plot(hist_baseline['val_iou'], label='Baseline UNet+BCE',          linewidth=2)
ax.plot(hist_attn['val_iou'],     label='AttnUNet+BCE+Dice', linewidth=2)
ax.set_title('IoU en validación'); ax.set_xlabel('Época'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1,0]
ax.plot(hist_baseline['val_f1'], label='Baseline UNet+BCE',          linewidth=2)
ax.plot(hist_attn['val_f1'],     label='AttnUNet+BCE+Dice', linewidth=2)
ax.set_title('F1/Dice en validación'); ax.set_xlabel('Época'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1,1]
us_b  = [m['umbral']  for m in tabla_baseline]
f1s_b = [m['f1_dice'] for m in tabla_baseline]
us_a  = [m['umbral']  for m in tabla_attn]
f1s_a = [m['f1_dice'] for m in tabla_attn]
ax.plot(us_b, f1s_b, 'o-', label='Baseline UNet+BCE',          linewidth=2)
ax.plot(us_a, f1s_a, 's-', label='AttnUNet+BCE+Dice', linewidth=2)
ax.axvline(umbral_baseline, color='C0', linestyle=':', alpha=0.5)
ax.axvline(umbral_attn,     color='C1', linestyle=':', alpha=0.5)
ax.set_title('F1 vs umbral (validación)'); ax.set_xlabel('Umbral'); ax.set_ylabel('F1')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/curvas_aprendizaje.png', dpi=120, bbox_inches='tight')
plt.show()

## Celda 23 — Resumen final

In [ ]:
print('=' * 80)
print('RESUMEN FINAL — TESIS')
print('=' * 80)
print(f"\nConfiguración:")
print(f"  - Datos:       Sentinel-2 mosaicos 2021 vs 2022, AOI Ucayali (~12,000 km²)")
print(f"  - Features:    12 canales (B2,B3,B4,B8,NDVI,BSI × 2 fechas)")
print(f"  - Etiquetas:   MapBiomas Amazonía Col 5 (deforestación 2021→2022)")
print(f"  - Splits:      {len(rutas_train)} train / {len(rutas_val)} val / {len(rutas_test)} test")
print(f"  - Épocas:      50")
print(f"  - Batch size:  {BATCH}")

print(f"\nResultados con umbral óptimo (validación):")
print(f"  Baseline U-Net + BCE  →  umbral {umbral_baseline:.2f}")
print(f"    F1: {m_baseline_opt['f1_dice']:.4f} | IoU: {m_baseline_opt['iou']:.4f} | "
      f"P: {m_baseline_opt['precision']:.3f} | R: {m_baseline_opt['recall']:.3f}")
print(f"  Attention U-Net + BCE+Dice  →  umbral {umbral_attn:.2f}")
print(f"    F1: {m_attn_opt['f1_dice']:.4f} | IoU: {m_attn_opt['iou']:.4f} | "
      f"P: {m_attn_opt['precision']:.3f} | R: {m_attn_opt['recall']:.3f}")
print(f"\nMejora del modelo propuesto:")
print(f"  ΔF1:  {m_attn_opt['f1_dice']-m_baseline_opt['f1_dice']:+.4f} "
      f"({100*(m_attn_opt['f1_dice']-m_baseline_opt['f1_dice'])/max(m_baseline_opt['f1_dice'],1e-6):+.1f}%)")
print(f"  ΔIoU: {m_attn_opt['iou']-m_baseline_opt['iou']:+.4f} "
      f"({100*(m_attn_opt['iou']-m_baseline_opt['iou'])/max(m_baseline_opt['iou'],1e-6):+.1f}%)")

In [ ]:
import numpy as np
import torch

def buscar_umbral_extendido(modelo, ruta_pesos, nombre):
    umbrales = np.arange(0.10, 0.96, 0.05)
    modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
    modelo.to(device).eval()

    todos_logits, todos_y = [], []
    with torch.no_grad():
        for x, y in dl_val:
            x, y = x.to(device), y.to(device)
            todos_logits.append(torch.sigmoid(modelo(x)).cpu())
            todos_y.append(y.cpu())
    probs = torch.cat(todos_logits)
    ys    = torch.cat(todos_y)

    print(f'\n=== {nombre} ===')
    print(f'{"Umbral":>8} {"P":>8} {"R":>8} {"F1":>8} {"IoU":>8}')
    mejor_f1 = 0; mejor_u = 0
    for u in umbrales:
        pred = (probs > u).float()
        tp = ((pred==1)&(ys==1)).sum().item()
        fp = ((pred==1)&(ys==0)).sum().item()
        fn = ((pred==0)&(ys==1)).sum().item()
        tn = ((pred==0)&(ys==0)).sum().item()
        eps = 1e-7
        p = tp/(tp+fp+eps); r = tp/(tp+fn+eps)
        f1 = 2*p*r/(p+r+eps)
        iou = tp/(tp+fp+fn+eps)
        marker = ' ← mejor' if f1 > mejor_f1 else ''
        if f1 > mejor_f1:
            mejor_f1 = f1; mejor_u = u
        print(f'{u:>8.2f} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {iou:>8.4f}{marker}')
    return mejor_u, mejor_f1

u_b, f1_b = buscar_umbral_extendido(
    UNet(c_in=12, c_out=1, base=32), ruta_baseline, 'Baseline UNet+BCE'
)
u_a, f1_a = buscar_umbral_extendido(
    AttentionUNet(c_in=12, c_out=1, base=32), ruta_attn, 'AttnUNet+BCE+Dice'
)

print(f'\nBaseline  → umbral óptimo: {u_b:.2f} | F1: {f1_b:.4f}')
print(f'AttnUNet  → umbral óptimo: {u_a:.2f} | F1: {f1_a:.4f}')

In [ ]:
import torch
import numpy as np

def buscar_umbral_extendido(modelo, ruta_pesos, nombre):
    umbrales = np.arange(0.10, 0.96, 0.05)
    modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
    modelo.to(device).eval()

    todos_logits, todos_y = [], []
    with torch.no_grad():
        for x, y in dl_val:
            x, y = x.to(device), y.to(device)
            todos_logits.append(torch.sigmoid(modelo(x)).cpu())
            todos_y.append(y.cpu())
    probs = torch.cat(todos_logits)
    ys    = torch.cat(todos_y)

    print(f'\n=== {nombre} ===')
    print(f'{"Umbral":>8} {"P":>8} {"R":>8} {"F1":>8} {"IoU":>8}')
    mejor_f1 = 0; mejor_u = 0
    for u in umbrales:
        pred = (probs > u).float()
        tp = ((pred==1)&(ys==1)).sum().item()
        fp = ((pred==1)&(ys==0)).sum().item()
        fn = ((pred==0)&(ys==1)).sum().item()
        tn = ((pred==0)&(ys==0)).sum().item()
        eps = 1e-7
        p = tp/(tp+fp+eps); r = tp/(tp+fn+eps)
        f1 = 2*p*r/(p+r+eps)
        iou = tp/(tp+fp+fn+eps)
        marker = ' ← mejor' if f1 > mejor_f1 else ''
        if f1 > mejor_f1:
            mejor_f1 = f1; mejor_u = u
        print(f'{u:>8.2f} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {iou:>8.4f}{marker}')
    return mejor_u, mejor_f1

u_b, f1_b = buscar_umbral_extendido(
    UNet(c_in=12, c_out=1, base=32), ruta_baseline, 'Baseline UNet+BCE'
)
u_a, f1_a = buscar_umbral_extendido(
    AttentionUNet(c_in=12, c_out=1, base=32), ruta_attn, 'AttentionUNet+BCE+Dice'
)

print(f'\nBaseline  → umbral óptimo: {u_b:.2f}, F1: {f1_b:.4f}')
print(f'AttnUNet  → umbral óptimo: {u_a:.2f}, F1: {f1_a:.4f}')

In [ ]:
# Analiza distribución de positivos por parche
import numpy as np, glob

positivos_por_parche = []
for split in ['train', 'val', 'test']:
    for r in sorted(glob.glob(f'/content/parches/{split}/*.npy')):
        arr = np.load(r, allow_pickle=False)
        pct = arr[12].mean() * 100
        positivos_por_parche.append(pct)

positivos_por_parche = np.array(positivos_por_parche)
print(f'Total parches: {len(positivos_por_parche)}')
print(f'Parches con 0% positivos:    {(positivos_por_parche == 0).sum()}')
print(f'Parches con <1% positivos:   {(positivos_por_parche < 1).sum()}')
print(f'Parches con 1-5% positivos:  {((positivos_por_parche>=1)&(positivos_por_parche<5)).sum()}')
print(f'Parches con 5-20% positivos: {((positivos_por_parche>=5)&(positivos_por_parche<20)).sum()}')
print(f'Parches con >20% positivos:  {(positivos_por_parche >= 20).sum()}')
print(f'\n% positivo promedio: {positivos_por_parche.mean():.3f}%')
print(f'% positivo máximo:   {positivos_por_parche.max():.3f}%')

In [ ]:
import numpy as np, glob, os, shutil

# Parámetros de rebalanceo
MIN_PCT_PARA_INCLUIR = 0.5   # elimina parches con <0.5% positivos
MAX_NEGATIVOS_PUROS  = 100   # conserva solo 100 parches casi vacíos (contexto)

dir_parches = '/content/parches'
eliminados = 0
conservados = 0
negativos_conservados = 0

for split in ['train', 'val', 'test']:
    rutas = sorted(glob.glob(f'{dir_parches}/{split}/*.npy'))
    for r in rutas:
        try:
            arr = np.load(r, allow_pickle=False)
            pct = arr[12].mean() * 100
            if pct == 0:
                # Parche completamente vacío — elimina casi todos
                if negativos_conservados < MAX_NEGATIVOS_PUROS:
                    negativos_conservados += 1
                    conservados += 1
                else:
                    os.remove(r)
                    eliminados += 1
            elif pct < MIN_PCT_PARA_INCLUIR:
                # Parche casi vacío — elimina
                os.remove(r)
                eliminados += 1
            else:
                conservados += 1
        except Exception:
            os.remove(r)
            eliminados += 1

print(f'Eliminados: {eliminados}')
print(f'Conservados: {conservados}')

# Verificar nuevo balance
for split in ['train', 'val', 'test']:
    rutas = sorted(glob.glob(f'{dir_parches}/{split}/*.npy'))
    pcts = [np.load(r, allow_pickle=False)[12].mean()*100 for r in rutas]
    print(f'{split}: {len(rutas)} parches | %pos promedio: {np.mean(pcts):.2f}%')

In [ ]:
import numpy as np, glob, os

MIN_PCT = 2.0  # solo parches con >2% de deforestación
eliminados = 0

for split in ['train', 'val', 'test']:
    for r in sorted(glob.glob(f'/content/parches/{split}/*.npy')):
        try:
            arr = np.load(r, allow_pickle=False)
            if arr[12].mean() * 100 < MIN_PCT:
                os.remove(r); eliminados += 1
        except:
            os.remove(r); eliminados += 1

print(f'Eliminados: {eliminados}')
for split in ['train','val','test']:
    rutas = glob.glob(f'/content/parches/{split}/*.npy')
    pcts = [np.load(r,allow_pickle=False)[12].mean()*100 for r in rutas]
    print(f'{split}: {len(rutas)} parches | %pos: {np.mean(pcts):.2f}%')

---